# Data Cleansing

## Import all requirements

In [40]:
import pandas as pd
import numpy as np
import missingno as msgn

url = '../data/airbnb.csv'
airbnb = pd.read_csv(url)

## Price fixing
### Remove dollar sign

In [41]:
# airbnb[~airbnb['price'].str.contains('$', regex=False, na=False)]
# airbnb['price'] = airbnb['price'].str.slice(0,-1)
airbnb['price'] = airbnb['price'].str.replace('$', '', regex=False)

### Convert prices to numeric values FLOAT

In [42]:
airbnb['price'] = pd.to_numeric(airbnb['price'])
airbnb['price'] = airbnb['price'].fillna(airbnb['price'].median())

In [43]:
airbnb.rename(columns={'Unnamed: 0' : 'ID'}, inplace=True)

## Coordinates cleaning

In [44]:
coords = airbnb['coordinates'].str.slice(1, -1).str.partition(',').drop(columns=1)
airbnb['longitude'] = coords[0]
airbnb['latitude'] = coords[2]
airbnb.drop(columns='coordinates', inplace=True)
airbnb['room_type'].unique()

<StringArray>
[        'Private room',      'Entire home/apt',              'Private',
          'Shared room',         'PRIVATE ROOM',                 'home',
 '   Shared room      ']
Length: 7, dtype: str

## Cleaning of the room types

In [45]:
airbnb['room_type'].unique()
# airbnb[airbnb['room_type'] == 'home']['room_type'] = 'Entire home/apt'
# airbnb[airbnb['room_type'] == 'home']['room_type'] = 'Entire home/apt'
for x in airbnb.index:
    if airbnb.loc[x, 'room_type'] == 'home':
        airbnb.loc[x, 'room_type'] = 'Entire home/apt'
    elif airbnb.loc[x, 'room_type'] == 'Private' or airbnb.loc[x, 'room_type'] == 'PRIVATE ROOM':
        airbnb.loc[x, 'room_type'] = 'Private room'
    elif airbnb.loc[x, 'room_type'] == '   Shared room      ':
        airbnb.loc[x, 'room_type'] = 'Shared room'

airbnb['room_type'].unique()

<StringArray>
['Private room', 'Entire home/apt', 'Shared room']
Length: 3, dtype: str

In [46]:
airbnb.isna().any()
# we will use mode to replace nan values

ID                    False
listing_id            False
name                   True
host_id               False
host_name              True
neighbourhood_full    False
room_type             False
price                 False
number_of_reviews     False
last_review            True
reviews_per_month      True
availability_365      False
rating                 True
number_of_stays        True
5_stars                True
listing_added         False
longitude             False
latitude              False
dtype: bool

In [47]:
airbnb['listing_added'] = pd.to_datetime(airbnb['listing_added'])
airbnb.isna().any()

ID                    False
listing_id            False
name                   True
host_id               False
host_name              True
neighbourhood_full    False
room_type             False
price                 False
number_of_reviews     False
last_review            True
reviews_per_month      True
availability_365      False
rating                 True
number_of_stays        True
5_stars                True
listing_added         False
longitude             False
latitude              False
dtype: bool

### Mark not visited places

In [48]:
airbnb['last_review'] = pd.to_datetime(airbnb['last_review'])

airbnb['visited'] = ~airbnb['last_review'].isna()

In [49]:
airbnb

,ID,listing_id,name,host_id,host_name,neighbourhood_full,room_type,price,number_of_reviews,last_review,reviews_per_month,availability_365,rating,number_of_stays,5_stars,listing_added,longitude,latitude,visited
0,0,13740704,"Cozy,budget friendly, cable inc, private entra...",20583125,Michel,"Brooklyn, Flatlands",Private room,45.0,10,2018-12-12,0.70,85,4.100954,12.0,0.609432,2018-06-08,40.63222,-73.93398,True
1,1,22005115,Two floor apartment near Central Park,82746113,Cecilia,"Manhattan, Upper West Side",Entire home/apt,135.0,1,2019-06-30,1.00,145,3.367600,1.2,0.746135,2018-12-25,40.78761,-73.96862,True
2,2,21667615,Beautiful 1BR in Brooklyn Heights,78251,Leslie,"Brooklyn, Brooklyn Heights",Entire home/apt,150.0,0,NaT,NaN,65,NaN,NaN,NaN,2018-08-15,40.7007,-73.99517,False
3,3,6425850,"Spacious, charming studio",32715865,Yelena,"Manhattan, Upper West Side",Entire home/apt,86.0,5,2017-09-23,0.13,0,4.763203,6.0,0.769947,2017-03-20,40.79169,-73.97498,True
4,4,22986519,Bedroom on the lively Lower East Side,154262349,Brooke,"Manhattan, Lower East Side",Private room,160.0,23,2019-06-12,2.29,102,3.822591,27.6,0.649383,2020-10-23,40.71884,-73.98354,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10014,10014,22307861,Lovely 1BR Harlem apartment,16004068,Rachel,"Manhattan, Harlem",Entire home/apt,105.0,4,2018-05-28,0.21,0,4.757555,4.8,0.639223,2017-11-22,40.80379,-73.95257,True
10015,10015,953275,Apartment For Your Holidays in NYC!,4460034,Alain,"Manhattan, East Harlem",Entire home/apt,125.0,50,2018-05-06,0.66,188,4.344704,60.0,0.648778,2017-10-31,40.79531,-73.9333,True
10016,10016,3452835,"Artsy, Garden Getaway in Central Brooklyn",666862,Amy,"Brooklyn, Clinton Hill",Entire home/apt,100.0,45,2016-11-27,0.98,0,3.966214,54.0,0.631713,2016-05-24,40.68266,-73.96743000000002,True
10017,10017,23540194,"Immaculate townhouse in Clinton Hill, Brooklyn",67176930,Sophie,"Brooklyn, Clinton Hill",Entire home/apt,450.0,2,2019-05-31,0.17,99,4.078581,2.4,0.703360,2018-11-25,40.68832,-73.96366,True


In [63]:
# airbnb['reviews_per_month'].fillna(airbnb['reviews_per_month'].median())
# airbnb.isna().any()

airbnb['reviews_per_month'] = airbnb['reviews_per_month'].fillna(airbnb['number_of_reviews'])

In [64]:
airbnb.isna().any()

ID                    False
listing_id            False
name                   True
host_id               False
host_name              True
neighbourhood_full    False
room_type             False
price                 False
number_of_reviews     False
last_review            True
reviews_per_month     False
availability_365      False
rating                 True
number_of_stays        True
5_stars                True
listing_added         False
longitude             False
latitude              False
visited               False
dtype: bool